# 01 — Temperature Interpolation

Three-step workflow:

| Step | Parameter | This notebook |
|------|-----------|---------------|
| 1 | `data=` | Meteostat weather API (or offline synthetic) |
| 2 | `boundary=` | Named place or 4-corner bbox |
| 3 | `method=` | IDW · Kriging · Spline · Natural Neighbor |

**Offline cells** run without any network access.  
**🌐 Network cells** fetch live Meteostat data.  
**🗺 Interactive maps** use [geemap](https://geemap.org).

In [ ]:
%pip install -q "geointerpo[full]" geemap localtileserver

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import matplotlib.pyplot as plt
import pandas as pd
from geointerpo import Pipeline

## Configuration

In [ ]:
BOUNDARY  = (-120.0, 49.0, -110.0, 60.0)  # Alberta, Canada  (min_lon, min_lat, max_lon, max_lat)
PLACE     = 'Alberta, Canada'              # 🌐 resolves to actual province boundary via Nominatim
DATE      = '2024-07-15'
RESOLUTION = 0.25  # degrees (~25 km)

---
## Part A — Offline demo (no network needed)

Uses built-in synthetic station data.  Good for exploring methods and parameters.

In [ ]:
result = Pipeline(
    data='sample',                  # Step 1 — built-in synthetic data
    variable='temperature',
    boundary=BOUNDARY,              # Step 2 — four-corner bbox
    method=['idw', 'kriging', 'spline', 'natural_neighbor'],  # Step 3
    method_params={
        'idw':    {'power': 2},
        'kriging': {'variogram_model': 'spherical'},
        'spline': {'spline_type': 'regularized'},
    },
    resolution=RESOLUTION,
    cv_folds=5,
).run()

In [ ]:
print('Cross-validation metrics:')
result.metrics_table()

In [ ]:
result.plot()
plt.suptitle('Temperature — Alberta, Canada (synthetic demo)', y=1.02)
plt.show()

### Station observations

In [ ]:
ax = result.stations.plot(
    column='value', cmap='RdYlBu_r', legend=True,
    figsize=(9, 6), markersize=40, edgecolor='k', linewidth=0.4
)
ax.set_title('Station observations — Alberta (synthetic, °C)')
plt.tight_layout()

---
## Part B — 🌐 Live Meteostat data

Fetches real daily weather station data.  Requires network access.

In [ ]:
%pip install -q meteostat

In [ ]:
result_live = Pipeline(
    data='meteostat',
    variable='temperature',
    date=DATE,
    boundary=PLACE,                # clips to actual Alberta polygon
    method=['idw', 'kriging', 'spline', 'natural_neighbor'],
    method_params={
        'kriging': {'variogram_model': 'spherical'},
    },
    resolution=RESOLUTION,
    cv_folds=5,
).run()

print(f'Loaded {len(result_live.stations)} real weather stations')

In [ ]:
result_live.metrics_table()

In [ ]:
result_live.plot()
plt.suptitle(f'Tavg — Alberta {DATE}  (Meteostat)', y=1.02)
plt.show()

### Variogram (Kriging)

In [ ]:
%pip install -q pykrige

In [ ]:
from geointerpo import viz
from geointerpo.interpolators import KrigingInterpolator

kriging_model = KrigingInterpolator(variogram_model='spherical').fit(result_live.stations)
fig = viz.plot_variogram(kriging_model, title=f'Spherical variogram — Tavg {DATE}')
plt.show()

### Variogram model comparison — PyKrige

[PyKrige](https://github.com/GeoStat-Framework/PyKrige) supports six built-in variogram models: `linear`, `power`, `gaussian`, `spherical`, `exponential`, and `hole-effect`. Fitting each to the same station data and comparing blocked CV metrics helps select the best model for the dataset — the exponential model tends to perform well for temperature fields with gradual spatial transitions.

In [ ]:
from geointerpo.interpolators import KrigingInterpolator
import pandas as pd

rows = []
for model in ['spherical', 'exponential', 'gaussian', 'linear', 'power']:
    ki = KrigingInterpolator(variogram_model=model).fit(result.stations)
    cv = ki.cross_validate(result.stations, k=5)
    params = ki.variogram_parameters   # [psill, range, nugget] for most; [slope, nugget] for linear
    rows.append({
        'model':  model,
        'rmse':   round(cv['rmse'], 3),
        'r':      round(cv['r'],    3),
        'param0': round(float(params[0]), 4),
        'param1': round(float(params[1]), 4),
        'nugget': round(float(params[2]), 4) if len(params) > 2 else '-',
    })

pd.DataFrame(rows).set_index('model')

### Universal Kriging — regional drift term — PyKrige

[PyKrige's UniversalKriging](https://github.com/GeoStat-Framework/PyKrige) extends Ordinary Kriging by adding a deterministic drift term (regional-linear by default) before kriging the residuals. This is beneficial when a large-scale gradient is present — e.g. temperature decreasing with latitude or elevation. Use `method='uk'` in the Pipeline.

In [ ]:
result_uk = Pipeline(
    data='sample',
    variable='temperature',
    boundary=BOUNDARY,
    method=['kriging', 'uk'],
    method_params={
        'kriging': {'variogram_model': 'spherical'},
        'uk':      {'variogram_model': 'spherical'},
    },
    resolution=RESOLUTION,
    cv_folds=5,
).run()

print('Ordinary Kriging vs Universal Kriging (regional-linear drift):')
result_uk.metrics_table()

In [ ]:
result_uk.plot(cmap='RdYlBu_r')
plt.suptitle('OK vs UK — temperature Alberta (synthetic, °C)', y=1.02)
plt.show()

### With DEM covariate (elevation)

Adds SRTM elevation as a covariate for ML/Regression-Kriging methods.

In [ ]:
result_dem = Pipeline(
    data='sample',
    variable='temperature',
    boundary=BOUNDARY,
    method=['rf', 'rk'],           # ML methods use DEM as covariate
    include_dem=True,
    dem_source='synthetic',        # use 'gee' or 'srtm' with network
    resolution=RESOLUTION,
    cv_folds=5,
).run()

result_dem.metrics_table()

### Gaussian Process — prediction + uncertainty

GP returns both a mean surface and a standard-deviation (uncertainty) grid.
High uncertainty indicates areas far from any station.

In [ ]:
from geointerpo.interpolators import MLInterpolator

gp_model = MLInterpolator(method='gp').fit(result.stations)
gp_mean, gp_std = gp_model.predict_with_std(BOUNDARY, resolution=RESOLUTION)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
gp_mean.plot(ax=axes[0], cmap='RdYlBu_r')
axes[0].set_title('GP — Mean prediction (°C)')
gp_std.plot(ax=axes[1], cmap='Oranges')
axes[1].set_title('GP — Uncertainty (std dev °C)')
plt.suptitle('Gaussian Process — Alberta (synthetic data)', y=1.02)
plt.tight_layout()
plt.show()

### Interactive map

### City-level zoom — Edmonton & Calgary

Clip the interpolation to city boundaries to show local detail.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, city in zip(axes, ['Edmonton, Alberta', 'Calgary, Alberta']):
    city_result = Pipeline(
        data='sample',
        variable='temperature',
        boundary=city,
        method='kriging',
        method_params={'kriging': {'variogram_model': 'spherical'}},
        resolution=0.05,
        cv_folds=3,
    ).run()
    city_result.grid.plot(ax=ax, cmap='RdYlBu_r')
    city_result.stations.plot(ax=ax, color='k', markersize=20, zorder=5)
    ax.set_title(city.split(',')[0])

plt.suptitle('Temperature — city-level zoom (synthetic, °C)', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
import geemap, tempfile

with tempfile.NamedTemporaryFile(suffix=".tif", delete=False) as f:
    tmp = f.name
result.grid.rio.set_spatial_dims(x_dim="lon", y_dim="lat").rio.write_crs("EPSG:4326").rio.to_raster(tmp)

m = geemap.Map(center=[54.5, -115.0], zoom=6, ee_initialize=False)
m.add_raster(tmp, colormap="rdylbu_r", layer_name="Temperature")
m.add_layer_manager()
m

---
## Save outputs

In [ ]:
result.save('outputs/temperature', geotiff=True, plot=True)
print('Saved to outputs/temperature/')